In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt
import fasttext

In [ ]:
with open("eval.json", "r") as f:
    results = json.load(f)
    datasets = list(list(results.values())[0]["accuracies"].keys())
    n_models = len(results)

    fig, ax = plt.subplots(figsize=(25,5))
    for i, (model, result) in enumerate(results.items()):
        ax.bar(np.arange(len(datasets)) + (i-(0.5*(n_models-1)))/n_models, list(result["accuracies"].values()), width=0.9/n_models, label=model)

    ax.set_xticks(np.arange(len(datasets)))
    ax.set_xticklabels(datasets)
    ax.set_ylabel("Accuracy")
    ax.set_title("Evaluation on French Math Data")
    ax.legend()
    plt.show()

In [ ]:
classifier = fasttext.load_model(config.MODEL_PATHS[0]+"facebook/fasttext-language-identification")

with open("eval.json", "r") as f:
    results = json.load(f)
    datasets = list(list(results.values())[0]["accuracies"].keys())
    n_models = len(results)

    fig, ax = plt.subplots(figsize=(25,5))
    for i, (model, result) in enumerate(results.items()):
        languages = {}
        for dataset in datasets:
            languages_dataset = {}
            for sample in result["samples"][dataset]:
                solution = "assistant".join(sample["generation"].split("assistant")[1:])
                lang_pred = classifier.predict(solution.replace("\n", " "), k=3)
                for lang, prob in zip(lang_pred[0], lang_pred[1]):
                    lang = lang.split("__label__")[1]
                    if prob > 0.02:
                        if lang in languages_dataset:
                            languages_dataset[lang] += prob
                        else:
                            languages_dataset[lang] = prob
                        if lang in languages:
                            languages[lang] += prob
                        else:
                            languages[lang] = prob
            for lang in languages_dataset.keys():
                languages_dataset[lang] = languages_dataset[lang] / len(result["samples"][dataset])
            print(dataset, ":", languages_dataset)
        for lang in languages.keys():
            languages[lang] = languages[lang] / sum([len(result["samples"][dataset]) for dataset in datasets])

        ax.bar(np.arange(len(languages)) + (i-(0.5*(n_models-1)))/n_models, list(languages.values()), width=0.9/n_models, label=model)

    ax.set_xticks(np.arange(len(languages)))
    ax.set_xticklabels(languages)
    ax.set_ylabel("Accuracy")
    ax.set_title("Evaluation on French Math Data")
    ax.legend()
    plt.show()